# Garbage Classification — EDA
CSE 4830 Group 9

**Run all cells top to bottom.** When the Drive mount cell runs, click the link and authorize.

**Pre-requisite:** `archive.zip` must be in the root of your Google Drive.

In [ ]:
!git clone https://github.com/LevinMathew1/Garbage-CNN.git /content/Garbage-CNN 2>/dev/null || echo 'Already cloned'
%cd /content/Garbage-CNN
!pip install -q timm
print('Setup complete.')

In [ ]:
import zipfile
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

zip_src     = Path('/content/drive/MyDrive/archive.zip')
extract_dir = Path('/content/data')

if not zip_src.exists():
    raise FileNotFoundError('archive.zip not found in Google Drive root. Upload it to drive.google.com first.')

if not extract_dir.exists():
    print('Extracting dataset...')
    with zipfile.ZipFile(zip_src) as zf:
        zf.extractall(extract_dir)
    print('Done.')
else:
    print('Dataset already extracted.')

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None — go to Runtime > Change runtime type > T4')

In [ ]:
from pathlib import Path

IMG_EXTS = {'.jpg', '.jpeg', '.png'}

def find_data_dir(root):
    for candidate in [root] + sorted([p for p in root.iterdir() if p.is_dir()]):
        subdirs = sorted([p for p in candidate.iterdir() if p.is_dir()])
        if not subdirs:
            continue
        if set(s.name for s in subdirs) >= {'train'}:
            return candidate
        if any(f.suffix.lower() in IMG_EXTS for f in list(subdirs[0].iterdir())[:10]):
            return candidate
    raise RuntimeError(f'Cannot find class directories under {root}')

DATA_DIR    = find_data_dir(Path('/content/data'))
source      = DATA_DIR / 'train' if (DATA_DIR / 'train').is_dir() else DATA_DIR
class_dirs  = sorted([p for p in source.iterdir() if p.is_dir()])
class_names = [p.name for p in class_dirs]
all_imgs    = [[f for f in d.iterdir() if f.suffix.lower() in IMG_EXTS] for d in class_dirs]

print(f'DATA_DIR  : {DATA_DIR}')
print(f'Classes   : {class_names}')
print(f'Total imgs: {sum(len(i) for i in all_imgs)}')

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

counts = [len(imgs) for imgs in all_imgs]
for cls, cnt in zip(class_names, counts):
    print(f'  {cls:20s}: {cnt}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar_label(ax.bar(class_names, counts, color='steelblue'))
ax.set_xlabel('Class'); ax.set_ylabel('Count')
ax.set_title('Class Distribution — Garbage Classification Dataset')
plt.xticks(rotation=30, ha='right'); fig.tight_layout()
Path('outputs/figures').mkdir(parents=True, exist_ok=True)
fig.savefig('outputs/figures/class_distribution.png', dpi=150)
plt.show()
print('Saved: outputs/figures/class_distribution.png')

In [ ]:
from PIL import Image
import random

random.seed(42)
N   = 4
fig, axes = plt.subplots(len(class_names), N, figsize=(N * 2.5, len(class_names) * 2.5))
if len(class_names) == 1:
    axes = [axes]

for row, (imgs, name) in enumerate(zip(all_imgs, class_names)):
    sample = random.sample(imgs, min(N, len(imgs)))
    for col in range(N):
        ax = axes[row][col]
        if col < len(sample):
            ax.imshow(Image.open(sample[col]).convert('RGB'))
        ax.axis('off')
        if col == 0:
            ax.set_title(name, fontsize=9, loc='left')

fig.suptitle('Sample Images per Class', fontsize=12)
fig.tight_layout(); plt.show()

In [ ]:
import numpy as np

widths, heights = [], []
for imgs in all_imgs:
    for p in imgs[:200]:
        try:
            w, h = Image.open(p).size
            widths.append(w); heights.append(h)
        except Exception:
            pass

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.hist(widths,  bins=30, color='steelblue'); ax1.set_title('Width Distribution');  ax1.set_xlabel('px')
ax2.hist(heights, bins=30, color='salmon');    ax2.set_title('Height Distribution'); ax2.set_xlabel('px')
fig.tight_layout(); plt.show()
print(f'Width  mean={np.mean(widths):.0f}px  |  Height mean={np.mean(heights):.0f}px')
print(f'\nDATA_DIR for training: "{DATA_DIR}"')